# Clase 22: Clustering y segmentación

## ¿Cómo agrupamos datos sin etiquetas?

En esta clase vamos a aprender a encontrar grupos naturales dentro de los datos usando **K-Means**. Es como ordenar M&Ms por color sin que nadie te diga cuántos colores hay ni cómo se llaman — el algoritmo lo descubre solo.

In [ ]:
# Importar las librerías necesarias
# pandas y numpy para datos, sklearn para clustering, matplotlib para visualizar
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Configuración visual
plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 11
print('Librerías cargadas correctamente ✅')

## 1. Cargar y explorar el dataset

Vamos a trabajar con `ventas_tienda.csv`. Primero echamos un vistazo general para entender qué columnas tenemos disponibles.

In [ ]:
# Cargar el dataset de ventas
# Este dataset tiene información de productos vendidos en una tienda
df = pd.read_csv('datasets/ventas_tienda.csv')

print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
print(f'\nColumnas: {list(df.columns)}')
print(f'\nValores nulos:\n{df.isnull().sum()}')
df.head()

## 2. Preparar los datos: escalar es clave

K-Means usa distancias entre puntos. Si una columna va de 0 a 1 000 000 y otra de 0 a 10, la primera dominará todas las decisiones. **Escalar** pone a todas las variables en el mismo rango.

In [ ]:
# Seleccionar columnas numéricas para el clustering
# Ajusta estos nombres según las columnas reales del dataset
columnas_cluster = df.select_dtypes(include='number').columns.tolist()
print(f'Columnas numéricas disponibles: {columnas_cluster}')

# Usar las dos primeras columnas numéricas para la visualización 2D
col1, col2 = columnas_cluster[0], columnas_cluster[1]
X = df[[col1, col2]].dropna()

# Escalar con StandardScaler: media=0, desv. estándar=1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'\nDatos antes de escalar — {col1} media: {X[col1].mean():.2f}')
print(f'Datos después de escalar — {col1} media: {X_scaled[:, 0].mean():.4f}')
print(f'Forma del array escalado: {X_scaled.shape}')

## 3. Método del Codo — ¿cuántos clusters elegir?

Probamos K de 1 a 10 y graficamos la **inercia** (suma de distancias al cuadrado a los centroides). Buscamos el punto donde la curva deja de bajar tan rápido — eso es el "codo".

In [ ]:
# Calcular la inercia para distintos valores de K
# A mayor K, menor inercia — pero en algún punto ya no vale la pena
inercias = []
valores_k = range(1, 11)

for k in valores_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)

# Graficar el Método del Codo
plt.figure(figsize=(9, 4))
plt.plot(valores_k, inercias, marker='o', color='steelblue', linewidth=2)
plt.xlabel('Número de clusters (K)')
plt.ylabel('Inercia')
plt.title('Método del Codo — ¿cuántos clusters usar?')
plt.xticks(valores_k)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('Observa dónde la curva tiene el mayor "quiebre" (codo)')

## 4. Silhouette Score — confirmar la calidad

El Silhouette Score mide qué tan bien separados están los clusters. Un valor más alto (cerca de +1) indica clusters más definidos.

In [ ]:
# Calcular el Silhouette Score para K de 2 a 8
# (no tiene sentido para K=1, necesitamos al menos 2 grupos)
silhouettes = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    silhouettes[k] = silhouette_score(X_scaled, labels)
    print(f'K={k}: Silhouette Score = {silhouettes[k]:.3f}')

# Graficar
plt.figure(figsize=(8, 4))
plt.bar(list(silhouettes.keys()), list(silhouettes.values()),
        color='coral', edgecolor='black')
plt.xlabel('K')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score por número de clusters')
plt.axhline(y=0.5, color='green', linestyle='--', label='Umbral 0.5')
plt.legend()
plt.show()

mejor_k = max(silhouettes, key=silhouettes.get)
print(f'\nMejor K según Silhouette: {mejor_k} (score={silhouettes[mejor_k]:.3f})')

## 5. Entrenamos K-Means con el K elegido

Usaremos K=3 como ejemplo. Ajusta este valor según lo que encontraste en los pasos anteriores.

In [ ]:
# Entrenar el modelo K-Means con K=3
# random_state=42 garantiza que siempre obtenemos el mismo resultado
K = 3  # <- cambia este valor según tu análisis

modelo = KMeans(n_clusters=K, random_state=42, n_init=10)
modelo.fit(X_scaled)

# Guardar los resultados en el DataFrame original
X_resultado = X.copy()
X_resultado['cluster'] = modelo.labels_

centros_escalados = modelo.cluster_centers_
centros_originales = scaler.inverse_transform(centros_escalados)

print(f'Inercia final: {modelo.inertia_:.2f}')
print(f'Silhouette Score: {silhouette_score(X_scaled, modelo.labels_):.3f}')
print(f'\nConteo por cluster:')
print(X_resultado['cluster'].value_counts().sort_index())

## 6. Visualizar los clusters

Graficamos cada punto con un color diferente según su cluster, y marcamos los centroides con una X roja.

In [ ]:
# Visualización de los clusters en 2D
# El color de cada punto indica a qué cluster pertenece
colores = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
nombres_cluster = [f'Cluster {i}' for i in range(K)]

plt.figure(figsize=(9, 6))

for i in range(K):
    mask = X_resultado['cluster'] == i
    plt.scatter(
        X_resultado.loc[mask, col1],
        X_resultado.loc[mask, col2],
        c=colores[i], label=nombres_cluster[i],
        alpha=0.7, edgecolors='k', linewidths=0.3, s=50
    )

# Marcar los centroides
plt.scatter(
    centros_originales[:, 0], centros_originales[:, 1],
    marker='X', s=300, color='red',
    zorder=5, label='Centroides'
)

plt.xlabel(col1)
plt.ylabel(col2)
plt.title(f'K-Means con K={K} — Segmentación de ventas')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Interpretar los clusters: del número al significado

Los números son solo el principio. Ahora vamos a entender **qué representa** cada cluster calculando el perfil promedio de cada grupo.

In [ ]:
# Calcular el perfil promedio de cada cluster
# Esto nos dice cuál es el 'representante típico' de cada grupo
perfil = X_resultado.groupby('cluster').mean().round(2)
conteo = X_resultado['cluster'].value_counts().sort_index().rename('n_registros')

perfil_completo = perfil.join(conteo)
print('Perfil promedio por cluster:')
print(perfil_completo)

# Asignar nombres descriptivos según las características
# (ajusta los nombres según lo que ves en los datos)
nombres_negocio = {}
for i in range(K):
    nombres_negocio[i] = f'Segmento {i+1}:  (escribe un nombre aquí)'

print('\nNombres de negocio propuestos:')
for cluster, nombre in nombres_negocio.items():
    print(f'  Cluster {cluster}: {nombre}')

## 8. DBSCAN: otro enfoque para encontrar grupos

DBSCAN no necesita que le digas cuántos clusters hay. Los detecta automáticamente según la densidad de puntos. Los outliers los marca con el número -1.

In [ ]:
# Aplicar DBSCAN al mismo dataset
# eps: radio de búsqueda de vecinos
# min_samples: mínimo de vecinos para ser núcleo de un cluster
db = DBSCAN(eps=0.5, min_samples=5)
db.fit(X_scaled)

n_clusters_db = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
n_ruido = list(db.labels_).count(-1)

print(f'Clusters encontrados por DBSCAN: {n_clusters_db}')
print(f'Puntos de ruido (outliers): {n_ruido} ({n_ruido/len(db.labels_)*100:.1f}%)')

# Visualizar
plt.figure(figsize=(9, 5))
scatter = plt.scatter(
    X[col1], X[col2],
    c=db.labels_, cmap='tab10',
    alpha=0.7, edgecolors='k', linewidths=0.2
)
plt.colorbar(scatter, label='Cluster (-1 = outlier)')
plt.xlabel(col1)
plt.ylabel(col2)
plt.title(f'DBSCAN — {n_clusters_db} clusters, {n_ruido} outliers detectados')
plt.tight_layout()
plt.show()

## 9. Aplicar al dataset de estudiantes

Ahora repite el análisis con `estudiantes.csv`. ¿Qué perfiles de rendimiento académico podemos identificar?

In [ ]:
# Cargar y explorar el dataset de estudiantes
# Buscamos grupos de estudiantes con rendimiento similar
df_est = pd.read_csv('datasets/estudiantes.csv')
print(f'Filas: {df_est.shape[0]} | Columnas: {df_est.shape[1]}')
print(f'\nColumnas: {list(df_est.columns)}')
df_est.head()

In [ ]:
# Seleccionar columnas numéricas del dataset de estudiantes
# Escalar, aplicar Método del Codo y K-Means
cols_est = df_est.select_dtypes(include='number').columns.tolist()
X_est = df_est[cols_est].dropna()

scaler_est = StandardScaler()
X_est_scaled = scaler_est.fit_transform(X_est)

# Método del Codo
inercias_est = []
for k in range(1, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_est_scaled)
    inercias_est.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 9), inercias_est, marker='o', color='green')
plt.xlabel('K')
plt.ylabel('Inercia')
plt.title('Método del Codo — Dataset de Estudiantes')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Entrenar K-Means y visualizar los segmentos de estudiantes
K_est = 3  # ajusta según el codo

modelo_est = KMeans(n_clusters=K_est, random_state=42, n_init=10)
modelo_est.fit(X_est_scaled)

df_est_result = X_est.copy()
df_est_result['cluster'] = modelo_est.labels_

# Perfil de estudiantes por cluster
perfil_est = df_est_result.groupby('cluster').mean().round(2)
conteo_est = df_est_result['cluster'].value_counts().sort_index()

print('Perfil promedio por cluster (estudiantes):')
print(perfil_est)
print(f'\nConteo: {conteo_est.to_dict()}')

# Visualización con las dos primeras columnas
c1_est, c2_est = cols_est[0], cols_est[1]
plt.figure(figsize=(9, 5))
for i in range(K_est):
    mask = df_est_result['cluster'] == i
    plt.scatter(
        df_est_result.loc[mask, c1_est],
        df_est_result.loc[mask, c2_est],
        label=f'Cluster {i}', alpha=0.7
    )
plt.xlabel(c1_est)
plt.ylabel(c2_est)
plt.title(f'Segmentación de Estudiantes — K={K_est}')
plt.legend()
plt.show()
print('Completa: ¿qué nombre le darías a cada cluster?')